In [71]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# LFRベンチマークグラフの生成パラメータ
n = 5000            # ノード数
tau1 = 3.0           # 次数分布の指数
tau2 = 1.016         # コミュニティサイズ分布の指数
mu = 0.8           # ミキシングパラメータ (小さいほどコミュニティが明確)
average_degree = 10 # 平均次数
min_community = 100 # 最小コミュニティサイズ
max_community = 1500 # 最大コミュニティサイズ

# グラフの生成
# seedを固定して再現性を確保します
G = nx.LFR_benchmark_graph(
    n, tau1, tau2, mu,
    average_degree=average_degree,
    min_community=min_community,
    max_community=max_community,
    seed=6
)

# 重複エッジの削除（LFR生成時に発生する場合があるため）
G = nx.Graph(G)
G.remove_edges_from(nx.selfloop_edges(G))

In [61]:
def get_ground_truth_map(G):
    """LFRグラフから正解コミュニティのリストと、ノードごとの所属辞書を抽出する"""
    # 各ノードの 'community' 属性から一意なコミュニティを抽出
    raw_communities = {frozenset(G.nodes[v]['community']) for v in G}
    communities = [list(c) for c in raw_communities]

    # ノードをキー、コミュニティIDを値とする辞書（メンバーシップ・マップ）を作成
    node_map = {node: idx for idx, members in enumerate(communities) for node in members}

    print(f"[Info] {len(communities)} 個のコミュニティを抽出しました。")
    return communities, node_map

# 実行例
communities, node_comm_map = get_ground_truth_map(G)

# サンプル出力
print("Node to Community ID map (sample):", dict(list(node_comm_map.items())[:5]))

[Info] 10 個のコミュニティを抽出しました。
Node to Community ID map (sample): {2052: 0, 8: 0, 9: 0, 2068: 0, 2070: 0}


In [ ]:
# 3. グラフの可視化
# 所属するコミュニティごとに色分けします
node_colors = np.zeros(G.number_of_nodes())
for comm_idx, community in enumerate(communities):
    for node in community:
        node_colors[node] = comm_idx

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=50, node_color=node_colors, cmap=plt.cm.rainbow)
nx.draw_networkx_edges(G, pos, alpha=0.1)
plt.title(f"LFR Benchmark Graph (Communities: {len(communities)})")
plt.axis('off')
plt.show()

In [72]:
def rank_within_community(G, comm_members, method="internal_degree"):
    """comm: {node: community_id}, c: 対象コミュニティ。スコア降順=コア。"""
    # members = [n for n in G if comm[n] == c]
    sub = G.subgraph(comm_members)
    if method == "internal_degree":           # 内部次数: 最も透明
        score = {n: sub.degree(n) for n in comm_members}
    elif method == "zscore":                  # within-module degree z-score (Guimerà–Amaral)
        d = np.array([sub.degree(n) for n in comm_members], float)
        z = (d - d.mean()) / (d.std() + 1e-12)
        score = dict(zip(comm_members, z))         # 単一コミュニティ内では内部次数と同順
    elif method == "subgraph_eigenvector":    # 中心メンバーと繋がるほど高い(再帰的)
        score = nx.eigenvector_centrality_numpy(sub)
    elif method == "ppr":                     # コミュニティからのPersonalized PageRank(全グラフ上)
        pers = {n: 1.0 for n in comm_members}
        pr = nx.pagerank(G, personalization=pers)
        score = {n: pr[n] for n in comm_members}
    return sorted(comm_members, key=score.get, reverse=True), score

In [74]:
rank_within_community(G, communities[2], method="ppr")[1]

{4097: 0.0009634327311085406,
 4101: 0.0006684422311921195,
 2054: 0.0003297066535287882,
 7: 0.00038002026561504093,
 4106: 0.00034632632452729945,
 14: 0.0003859423691399774,
 18: 0.0008378176751960937,
 4115: 0.00042072989665600336,
 2067: 0.0003518207480606605,
 20: 0.00029505304398290736,
 2072: 0.0003145652285460084,
 2074: 0.0007916032178390914,
 29: 0.0003579980418517459,
 2080: 0.0003803357327785429,
 2083: 0.00033628645971955744,
 36: 0.000436536567500697,
 4135: 0.00026940685256611577,
 42: 0.0003266919563844114,
 2096: 0.00040151508102152913,
 52: 0.0003862988451566098,
 60: 0.0003201551960269258,
 2109: 0.00035348127832029015,
 2111: 0.0003587405437354114,
 4161: 0.00033221934197464025,
 2118: 0.00039150360431062717,
 4176: 0.00026171488260509277,
 89: 0.0003202051758000805,
 2138: 0.00038178765377432165,
 92: 0.0002787855283721902,
 102: 0.0004504608220139901,
 2165: 0.0003371633323113071,
 122: 0.00036889605906468073,
 4222: 0.00032014496306361347,
 2175: 0.0004505967728

In [75]:
import pandas as pd

results = []

for i, comm_members in enumerate(communities):
    # 各コミュニティに対してPPRスコアを取得
    _, scores = rank_within_community(G, comm_members, method="ppr")
    
    # スコアのリストを作成
    score_values = list(scores.values())
    
    # 統計量の計算
    comm_sum = np.sum(score_values)
    comm_mean = np.mean(score_values)
    
    results.append({
        "Community ID": i,
        "Size": len(comm_members),
        "PPR Sum": comm_sum,
        "PPR Mean": comm_mean
    })

# 結果をデータフレームにまとめて表示
results_df = pd.DataFrame(results)
display(results_df)

,Community ID,Size,PPR Sum,PPR Mean
0,0,141,0.179787,0.001275
1,1,1082,0.334741,0.000309
2,2,960,0.349017,0.000364
3,3,198,0.199385,0.001007
4,4,131,0.179356,0.001369
5,5,224,0.194798,0.000870
6,6,128,0.179505,0.001402
7,7,283,0.192976,0.000682
8,8,1159,0.313634,0.000271
9,9,694,0.250568,0.000361


In [76]:
# PPR Meanが高い順（降順）にソート
sorted_results_df = results_df.sort_values(by='PPR Mean', ascending=False)
display(sorted_results_df)

,Community ID,Size,PPR Sum,PPR Mean
6,6,128,0.179505,0.001402
4,4,131,0.179356,0.001369
0,0,141,0.179787,0.001275
3,3,198,0.199385,0.001007
5,5,224,0.194798,0.000870
7,7,283,0.192976,0.000682
2,2,960,0.349017,0.000364
9,9,694,0.250568,0.000361
1,1,1082,0.334741,0.000309
8,8,1159,0.313634,0.000271
